<a href="https://colab.research.google.com/github/surajjeoor/masai_assignments/blob/main/Regularization_techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#install minimums for reproducibility
!pip install -q -U "scikit-learn>=1.4" "plotly>=5.20" "joblib>=1.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 70.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "colab"

from sklearn.datasets import load_diabetes
from sklearn.model_selection import (
    train_test_split, KFold, cross_val_score, cross_validate, GridSearchCV,
)
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV,
)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import r2_score, mean_squared_error

In [3]:
#loading the dataset and doing statistical analysis
data=load_diabetes(as_frame=True)
X=data.data
y=data.target
feature_names=X.columns.to_list()
print(f"Columns in the dataframe : {feature_names}")
print(f"Target range :{y.min():.2f} to {y.max():.2f},and the mean value of the target variable is :{y.mean()}")

print("\n\n overall statistical analysis can be shown as follows:\n")
X.describe()

Columns in the dataframe : ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Target range :25.00 to 346.00,and the mean value of the target variable is :152.13348416289594


 overall statistical analysis can be shown as follows:



,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
count,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02
mean,-2.511817e-19,1.230790e-17,-2.245564e-16,-4.797570e-17,-1.381499e-17,3.918434e-17,-5.777179e-18,-9.042540e-18,9.268604e-17,1.130318e-17
std,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02
min,-1.072256e-01,-4.464164e-02,-9.027530e-02,-1.123988e-01,-1.267807e-01,-1.156131e-01,-1.023071e-01,-7.639450e-02,-1.260971e-01,-1.377672e-01
25%,-3.729927e-02,-4.464164e-02,-3.422907e-02,-3.665608e-02,-3.424784e-02,-3.035840e-02,-3.511716e-02,-3.949338e-02,-3.324559e-02,-3.317903e-02
50%,5.383060e-03,-4.464164e-02,-7.283766e-03,-5.670422e-03,-4.320866e-03,-3.819065e-03,-6.584468e-03,-2.592262e-03,-1.947171e-03,-1.077698e-03
75%,3.807591e-02,5.068012e-02,3.124802e-02,3.564379e-02,2.835801e-02,2.984439e-02,2.931150e-02,3.430886e-02,3.243232e-02,2.791705e-02
max,1.107267e-01,5.068012e-02,1.705552e-01,1.320436e-01,1.539137e-01,1.987880e-01,1.811791e-01,1.852344e-01,1.335973e-01,1.356118e-01


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes

# Redefine X and y to ensure they are available in this cell
data = load_diabetes(as_frame=True)
X = data.data
y = data.target
feature_names = X.columns.to_list()
split_results=[]
for rs in [42,0,7,123]:
  x_tr,x_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,random_state=rs)
  model=LinearRegression()
  model.fit(x_tr,y_tr)
  y_pred=model.predict(x_te)
  r2=r2_score(y_te,y_pred)
  split_results.append(r2)
  print(f"The random state:{rs:.1f} -------> The r2 score is {r2:.3f}")

The random state:42.0 -------> The r2 score is 0.453
The random state:0.0 -------> The r2 score is 0.332
The random state:7.0 -------> The r2 score is 0.404
The random state:123.0 -------> The r2 score is 0.568


In [13]:
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print(f"training dataset size {x_train.shape}")
print(f"testing dataset size {x_test.shape}")

training dataset size (353, 10)
testing dataset size (89, 10)


In [14]:
#Baseline OLS CV score for training dataset
ols_pipe=Pipeline([("scaler",StandardScaler()),("model",LinearRegression())])
ols_cv_results=cross_validate(ols_pipe,x_train,y_train,cv=5,scoring="r2")
print(f"the range of crossvalidation scores is {np.min(ols_cv_results['test_score']):.3f} to {np.max(ols_cv_results['test_score']):.3f}")
print(f"the mean of crossvalidation scores is {np.mean(ols_cv_results['test_score']):.3f}")

the range of crossvalidation scores is 0.215 to 0.618
the mean of crossvalidation scores is 0.449


In [15]:
#Sweeps alpha for Lasso across np.logspace(-3, 1, 50). For each alpha, fit Lasso on the FULL training set, record the coefficients and the number of non-zero coefficients
alpha_range=np.logspace(-3, 1, 50)
lasso_coefs=[]
lasso_non_zero_coefs=[]
for alpha in alpha_range:
  lasso_pipe=Pipeline([("scaler",StandardScaler()),("model",Lasso(alpha=alpha))])
  lasso_pipe.fit(x_train,y_train)
  lasso_coefs.append(lasso_pipe.named_steps["model"].coef_)
  lasso_non_zero_coefs.append(np.sum(lasso_pipe.named_steps["model"].coef_!=0))

lasso_coefs=np.array(lasso_coefs)
lasso_non_zero_coefs=np.array(lasso_non_zero_coefs)
print(f"Lasso coefficients shape:{lasso_coefs.shape}(alphas x features)")
print(f"Lasso non-zero coefficients shape:{lasso_non_zero_coefs.shape}")

Lasso coefficients shape:(50, 10)(alphas x features)
Lasso non-zero coefficients shape:(50,)


In [17]:
#plot the graph for the lasso coefficients
fig=go.Figure()
for i,feat in enumerate(feature_names):
  fig.add_trace(go.Scatter(x=alpha_range,
                           y=lasso_coefs[:,i],
                           mode="lines",
                           name=f"coefficient for {feat}",
                           line=dict(width=2),
                           hovertemplate=f"<b>{feat}</b><br>alpha=%{{x:.3f}}<br>coef=%{{y:.3f}}<extra></extra>"

  ))
fig.update_layout(
    title="Ridge coefficient path on Diabetes — smooth shrinkage, never zero",
    xaxis_title="alpha (log scale)",
    yaxis_title="coefficient value",
    xaxis_type="log",
    paper_bgcolor="white", plot_bgcolor="white",
    width=850, height=500,
)
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.show()

Other than S2,S4 and S6, all of the coefficients are survivors coefficient.

In [18]:
MASAI_RED = "#ED1C24"

In [19]:
#count non-zero coefficients at each alpha and plot it
non_zero_counts=(np.abs(lasso_coefs)>1e-6).sum(axis=1)
fig=go.Figure(go.Scatter(x=alpha_range,
                         y=non_zero_counts,
                         mode="lines+markers",
                         line=dict(color=MASAI_RED, width=2),
                         hovertemplate="alpha=%{x:.2f}<br>features=%{y}<extra></extra>"
))
fig.update_layout(
    title="Number of non-zero Lasso coefficients vs alpha",
    xaxis_title="alpha (log scale)",
    yaxis_title="# non-zero coefficients",
    xaxis_type="log",
    paper_bgcolor="white", plot_bgcolor="white",
    width=750, height=400,
)
fig.show()
